In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.silver.silver_employee_assignments AS
WITH source_cleaned AS (
  SELECT 
    upper(trim(line_code)) AS line_code,
    trim(line_name) AS line_name,
    upper(trim(cell_code)) AS cell_code,
    trim(cell_name) AS cell_name,
    CAST(default_employee_key AS INT) AS default_employee_key,
    trim(default_employee_name) AS default_employee_name,
    CAST(backup_employee_key AS INT) AS backup_employee_key,
    trim(backup_employee_name) AS backup_employee_name,
    upper(trim(coalesce(shift, 'ALL SHIFTS'))) AS shift,
    CAST(valid_from AS DATE) AS valid_from,
    trim(remarks) AS remarks,
    trim(source) AS source,
    _source_file,
    _bronze_ingested_at
  FROM data_warehouse_factory.bronze.bronze_employee_assignments
  WHERE line_code IS NOT NULL AND trim(line_code) != ''
    AND cell_code IS NOT NULL AND trim(cell_code) != ''
    AND valid_from IS NOT NULL
),

-- 1. Deduplikacja: wybór najnowszego wpisu dla tej samej stacji, zmiany i daty obowiązywania
deduplicated AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY line_code, cell_code, shift, valid_from
      ORDER BY _bronze_ingested_at DESC
    ) AS rn
  FROM source_cleaned
),

-- 2. Logika SCD Type 2: wyznaczenie kolejnej daty zmiany obsady
scd2_prep AS (
  SELECT 
    *,
    LEAD(valid_from) OVER (
      PARTITION BY line_code, cell_code, shift 
      ORDER BY valid_from ASC
    ) AS next_valid_from
  FROM deduplicated
  WHERE rn = 1
)

-- 3. Zbudowanie gotowych interwałów ważności i kluczy
SELECT 
  md5(concat_ws('||', line_code, cell_code, shift, cast(valid_from as string))) AS assignment_key,
  md5(concat_ws('||', line_code, cell_code)) AS cell_key,
  line_code,
  line_name,
  cell_code,
  cell_name,
  default_employee_key,
  default_employee_name,
  backup_employee_key,
  backup_employee_name,
  shift,
  valid_from,
  COALESCE(DATE_ADD(next_valid_from, -1), DATE '9999-12-31') AS valid_to,
  CASE WHEN next_valid_from IS NULL THEN TRUE ELSE FALSE END AS is_current,
  remarks,
  source,
  _source_file,
  _bronze_ingested_at,
  current_timestamp() AS _silver_ingested_at
FROM scd2_prep;